# IOAI — 2025 Summer National Catacomb Adventure (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CreepyCatacombs Gymnasium 패키지는 노트북 첫 셀이 자동 설치합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Catacomb Adventure (Katakomba Kaland)

> **HAIO 2025 — Summer National Finals (RL)**

Reach the exit of the **CreepyCatacombs** Gymnasium maze (canonical **seed 2025**) with maximum reward. Score = **episode reward** of your policy on seed 2025 (higher is better; −1/step, −5 for bumping a wall, big bonus at the exit). **Submit** `submission.npy` — a **Q-table** `(n_states, n_actions)` (graded greedily by `argmax`).

Baseline: the env exposes `info['grid']` (−1 wall, 2 goal, −2 plothole) + agent/goal positions, so we **plan with BFS** and bake the shortest path into a Q-table. (Or learn it with RL — MC/SARSA.)

In [ ]:
import os, logging, warnings
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')   # 헤드리스(사운드카드 없음): ALSA 오디오 오류 억제
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')   # 오프스크린(rgb_array) 렌더는 그대로 동작
logging.disable(logging.INFO)                       # pygame 렌더러 INFO/DEBUG 로그 스팸 억제
warnings.filterwarnings('ignore', message='.*pkg_resources is deprecated.*')
import subprocess, sys, os
TGT='/workspace/.haio_pkgs' if os.path.isdir('/workspace') else '/tmp/haio_pkgs'
if not os.path.isdir(TGT):
    subprocess.run([sys.executable,'-m','pip','install','-q','--target',TGT,'creepy-catacombs-s1==0.1.4'],check=True)
sys.path.insert(0, TGT)
import creepy_catacombs_s1, gymnasium as gym, numpy as np
from collections import deque
SEED=2025
env=gym.make('CreepyCatacombs-v0'); o,info=env.reset(seed=SEED)
grid=np.array(info['grid']); H,W=grid.shape
start=tuple(int(x) for x in info['agent_pos']); goal=tuple(int(x) for x in info['goal_pos'])
nS=env.observation_space.n
print('grid',grid.shape,'start',start,'goal',goal,'states',nS)

## BFS shortest path → Q-table (actions 0=up,1=right,2=down,3=left)

In [ ]:
MOVES={0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
def bfs(avoid_plothole=True):
    seen={start}; parent={}; q=deque([start])
    while q:
        cur=q.popleft()
        if cur==goal: return parent
        for a,(dr,dc) in MOVES.items():
            nr,nc=cur[0]+dr,cur[1]+dc
            if 0<=nr<H and 0<=nc<W and grid[nr,nc]!=-1 and (not avoid_plothole or grid[nr,nc]!=-2) and (nr,nc) not in seen:
                seen.add((nr,nc)); parent[(nr,nc)]=(cur,a); q.append((nr,nc))
    return None
parent=bfs(True) or bfs(False)
Q=np.zeros((nS,4)); cur=goal
while cur!=start:
    prev,a=parent[cur]; Q[prev[0]*W+prev[1], a]=1.0; cur=prev
def run(seed):
    o,_=env.reset(seed=seed); tot=0
    for _ in range(300):
        o,r,d,t,_=env.step(int(np.argmax(Q[o]))); tot+=r
        if d or t: break
    return tot
print('self-check reward (seed 2025):', run(SEED))

## Save Q-table → `submission.npy`

In [ ]:
np.save('submission.npy', Q); print('wrote submission.npy', Q.shape)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)